# OPAL Fuels — Financial Data Pipeline

**Georgetown MSBA | FINC 6580: FinTech | Project 2**

This notebook pulls historical financial statements for OPAL Fuels (NASDAQ: OPAL) from Yahoo Finance,
reshapes them into a tidy long format, and exports them to Excel for use in the DCF valuation model.

OPAL Fuels is a renewable natural gas (RNG) producer that converts biogas from landfills, dairy farms,
and wastewater treatment plants into low-carbon fuel for heavy-duty transportation.

## 1.0 Setup

In [ ]:
# Import libraries
import yfinance as yf
import pandas as pd

## 2.0 Pull Financial Statements

`yfinance` returns financials in wide format — each fiscal year is a separate column.
We pull three statements: Income Statement, Cash Flow, and Balance Sheet.

In [ ]:
# Define company ticker
ticker = "OPAL"

# Pull company data
company = yf.Ticker(ticker)

# Get annual financial statements
income_stmt   = company.income_stmt.reset_index(names='measure')
cash_flow     = company.cashflow.reset_index(names='measure')
balance_sheet = company.balance_sheet.reset_index(names='measure')

## 3.0 Reshape to Long Format

Wide format (years as columns) is convenient for display but awkward for modeling.
We melt each statement into long format: one row per (measure, year) pair.
This makes it easy to filter, pivot, and chart individual line items in Excel or Python.

In [ ]:
def to_long(df):
    """Melt a wide financial statement DataFrame into long format."""
    return pd.melt(
        df,
        id_vars='measure',
        value_vars=df.columns[1:],
        var_name='year',
        value_name='value'
    )

income_stmt   = to_long(income_stmt)
cash_flow     = to_long(cash_flow)
balance_sheet = to_long(balance_sheet)

# Preview
income_stmt.head()

## 4.0 Export to Excel

Each statement is written to a separate sheet in `financials.xlsx`,
which feeds directly into the DCF model (`dcf_model.xlsx`).

In [ ]:
output_path = 'financials.xlsx'

with pd.ExcelWriter(output_path) as writer:
    income_stmt.to_excel(writer,   sheet_name='Income Statement', index=False)
    cash_flow.to_excel(writer,     sheet_name='Cash Flow',        index=False)
    balance_sheet.to_excel(writer, sheet_name='Balance Sheet',    index=False)

print(f"Exported financial statements to '{output_path}'")